In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.tsa.seasonal import STL

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().resolve().parent))

print(sys.executable)

from src.metrics import *
print("ok")

/Users/yuanxinyi/Desktop/panel-forcast-kit/.venv/bin/python
ok


In [4]:
project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "train.csv").exists()),
    Path.cwd(),
)
csv_path = project_root / "data" / "raw" / "train.csv"

df = pd.read_csv(
    csv_path,
    parse_dates=["date"],
) 

# Preparing Features

In [18]:
# 1) Calendar Feature
df["date"] = pd.to_datetime(df["date"]) 
df["day"]        = df["date"].dt.day 
df["month"]      = df["date"].dt.month 
df["year"]       = df["date"].dt.year
df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
df["dayofweek"]  = df["date"].dt.dayofweek
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)


# 2) Holiday Feature
holiday_dates = pd.to_datetime([
    "2013-01-01", "2014-01-01", "2015-01-01", "2016-01-01", "2017-01-01",
    "2013-12-25", "2014-12-25", "2015-12-25", "2016-12-25", "2017-12-25"
])
df["is_holiday"] = df["date"].isin(holiday_dates).astype(int)

# 3) Lag Feature (for daily total sales of each item in each store)
df = df.sort_values(["store", "item", "date"]).reset_index(drop=True)
for lag in [1, 7, 14, 28]:
    df[f"lag_{lag}"] = (
        df.groupby(["store", "item"])["sales"]
          .transform(lambda s: s.shift(lag))
    )

# 4) Rolling Feature (mean and std for daily total sales of each item in each store)
df["rolling_mean_7"] = (
    df.groupby(["store", "item"])["sales"]
      .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
)
df["rolling_std_7"] = (
    df.groupby(["store", "item"])["sales"]
      .transform(lambda s: s.shift(1).rolling(7, min_periods=1).std())
)

In [19]:
# 5) sanity check: if the features are correctly generated
# Calendar features
sample = df[(df["store"]==1) & (df["item"]==1)][["day","month","year","weekofyear","dayofweek","is_weekend","is_holiday"]].head(5)
print(sample.to_string(index=False))

# Lag and rolling features
sample = df[(df["store"]==1) & (df["item"]==1)][["lag_1", "lag_7","lag_14","lag_28","rolling_mean_7","rolling_std_7"]].head(5)
print("\n", sample.to_string(index=False))

 day  month  year  weekofyear  dayofweek  is_weekend  is_holiday
   1      1  2013           1          1           0           1
   2      1  2013           1          2           0           0
   3      1  2013           1          3           0           0
   4      1  2013           1          4           0           0
   5      1  2013           1          5           1           0

  lag_1  lag_7  lag_14  lag_28  rolling_mean_7  rolling_std_7
   NaN    NaN     NaN     NaN             NaN            NaN
  13.0    NaN     NaN     NaN       13.000000            NaN
  11.0    NaN     NaN     NaN       12.000000       1.414214
  14.0    NaN     NaN     NaN       12.666667       1.527525
  13.0    NaN     NaN     NaN       12.750000       1.258306


#  Building and Cutting Dataset

In [15]:
# Select features and drop rows with missing values in these features or in the target variable
feature_cols = [
    "store", "item", "day", "month", "is_weekend", "is_holiday",
    "lag_1", "lag_7", "lag_14", "lag_28", "rolling_mean_7"
]
model_df = df.dropna(subset=feature_cols + ["sales", "date"]).copy()
X = model_df[feature_cols]
y = model_df["sales"]

sample = X.head(15)
print(sample.to_string(index=False))
print(f"\n=== Current training set shape ===\n Training examples: {X.shape}, Target: {y.shape}")
print(f"\n=== Current training set time range ===\n Start: {model_df['date'].min()}, End: {model_df['date'].max()}")

 store  item  day  month  is_weekend  is_holiday  lag_1  lag_7  lag_14  lag_28  rolling_mean_7
     1     1   29      1           0           0   11.0    7.0     5.0    13.0       10.428571
     1     1   30      1           0           0    6.0    9.0     7.0    11.0       10.285714
     1     1   31      1           0           0    9.0    8.0    16.0    14.0       10.285714
     1     1    1      2           0           0   13.0   14.0     7.0    13.0       11.000000
     1     1    2      2           1           0   11.0   12.0    18.0    10.0       10.571429
     1     1    3      2           1           0   21.0   12.0    15.0    12.0       11.857143
     1     1    4      2           0           0   15.0   11.0     8.0    10.0       12.285714
     1     1    5      2           0           0   14.0    6.0     7.0     9.0       12.714286
     1     1    6      2           0           0    9.0    9.0     9.0    12.0       13.142857
     1     1    7      2           0           0  

In [22]:
# Split the dataset into training and testing sets based on a cutoff date
cutoff = pd.Timestamp("2017-10-01")
train_mask = model_df["date"] < cutoff
test_mask = model_df["date"] >= cutoff

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"\n=== Training set ===\n {model_df.loc[train_mask, 'date'].min().date()} ~ {model_df.loc[train_mask, 'date'].max().date()}"
      f"  | X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"\n=== Test set ===\n {model_df.loc[test_mask, 'date'].min().date()} ~ {model_df.loc[test_mask, 'date'].max().date()}"
      f"  | X_test: {X_test.shape}, y_test: {y_test.shape}")
sample_train = pd.concat([X_train.head(5), y_train.head(5)], axis=1)
print(f"\n=== Training sample ===\n{sample_train.to_string(index=False)}")
sample_test = pd.concat([X_test.head(5), y_test.head(5)], axis=1)
print(f"\n=== Test sample ===\n{sample_test.to_string(index=False)}")


=== Training set ===
 2013-01-29 ~ 2017-09-30  | X_train: (853000, 11), y_train: (853000,)

=== Test set ===
 2017-10-01 ~ 2017-12-31  | X_test: (46000, 11), y_test: (46000,)

=== Training sample ===
 store  item  day  month  is_weekend  is_holiday  lag_1  lag_7  lag_14  lag_28  rolling_mean_7  sales
     1     1   29      1           0           0   11.0    7.0     5.0    13.0       10.428571      6
     1     1   30      1           0           0    6.0    9.0     7.0    11.0       10.285714      9
     1     1   31      1           0           0    9.0    8.0    16.0    14.0       10.285714     13
     1     1    1      2           0           0   13.0   14.0     7.0    13.0       11.000000     11
     1     1    2      2           1           0   11.0   12.0    18.0    10.0       10.571429     21

=== Test sample ===
 store  item  day  month  is_weekend  is_holiday  lag_1  lag_7  lag_14  lag_28  rolling_mean_7  sales
     1     1    1     10           1           0   30.0   24.0  

In [21]:
# Save the generated datasets to the processed folder
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / "X_train.csv", index=False)
y_train.to_csv(processed_dir / "y_train.csv", index=False)
X_test.to_csv(processed_dir / "X_test.csv", index=False)
y_test.to_csv(processed_dir / "y_test.csv", index=False)
model_df.to_csv(processed_dir / "model_df.csv", index=False)

print(f"Saved files to: {processed_dir}")
print("Saved: X_train.csv, y_train.csv, X_test.csv, y_test.csv, model_df.csv")

Saved files to: /Users/yuanxinyi/Desktop/panel-forcast-kit/data/processed
Saved: X_train.csv, y_train.csv, X_test.csv, y_test.csv, model_df.csv
